# [0.0] DESI Data: Exploration
*DESI Data Release 1 (DR1) includes spectra for ~18 million unique targets from Main survey observations taken from May 2021 through June 2022, as well as a reprocessing of the Survey Validation data previously released in the DESI Early Data Release (EDR).*

In [3]:
import os
import fitsio
import numpy as np
from tqdm import tqdm
from pathlib import Path
from astropy.io import fits
from astropy.table import join, Table
import pandas as pd
import nested_pandas as npd
import matplotlib.pyplot as plt
import glob
import pyarrow as pa
import lsdb
from hats.io.validation import is_valid_catalog
from hats_import.catalog.file_readers import InputReader
from hats_import.pipeline import pipeline_with_client
from hats_import.catalog.arguments import ImportArguments
from dask.distributed import Client

/global/homes/o/olynn/.venvs/p312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
main_directory = Path("/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron")
core_redshift_catalog = main_directory / "zcatalog/v1/zall-pix-iron.fits"
spectra_directory = main_directory / "healpix/"

## 00 - Overview

Summary of DR1 (Iron) Main Survey:
- Number of useful(1) spectra: 18,659,804
  - Galaxies (SPECTYPE==GALAXY): 13,049,402
  - Quasars (SPECTYPE==QSO): 1,553,713
  - Stars (SPECTYPE==STAR): 4,056,689
- DESI Instrument 	
  - Spectral coverage(2): 	360–982.4 nm
  - Spectral resolution: 	2000 (at 360 nm) – 5500 (at 980 nm)
  - Wavelength system: 	vacuum barycentric
  - Photometric bands (Legacy Surveys DR9): 	g, r, z, W1, W2, W3, W4
- Approximate area 	
  - Main Survey / Backup: 	2,726 sq. deg.
  - Main Survey / Bright: 	9,739 sq. deg.
  - Main Survey / Dark: 	9,528 sq. deg.

*(1) “Useful spectra” are defined as having ZCAT_PRIMARY==True, OBJTYPE=='TGT', and ZWARN==0, which selects all unique, non-sky targets with no known redshift-fitting failures. The Main Survey additionally excludes PROGRAM=='other'. See dr1paper notebook.*

*(2) Spectra are split on three spectrograph arms: blue (B), red (R), infrared (Z).*

### Data Organization
**Redshift catalogs**

Redshift and classification catalogs are combined across thousands of individual files into stacked redshift catalogs in `spectro/redux/MOUNTAIN/zcatalog/`. For DR1, `MOUNTAIN=iron` (for EDR, `MOUNTAIN=fuji`).

Multiple zcatalog versions may be released with each spectroscopic production. In general, the highest version number is the preferred catalog. DR1 uses version 1 (v1) catalogs, and older v0 catalogs are deprecated. A planned future v2 will significantly reformat the files to make them easier to work with the rapidly increasing data volumes of the DESI catalogs.

For analyses that just want the recommended “best” redshift for a given target, regardless of survey or program, we recommend using:
- `zall-pix-iron.fits`: (20.8GB) Combines all the HEALPix-based redshifts across all surveys and programs. The ZCAT_PRIMARY boolean column indicates the recommended redshift.
- `zall-tilecumulative-iron.fits`: (23.6GB) Provides all cumulative, tile-based redshifts across all surveys and programs.

Note that these are v1 catalogs, which are the preferred version for DR1.

**HEALPixel-based spectra**

Full-depth coadds are located under `spectro/redux/MOUNTAIN/healpix/`, and combine exposures for targets on a given HEALPixel (nside = 64; see Górski et al. (2005)), including combining data across tiles if a target was observed on multiple tiles. These are further divided by *survey* and *program*, and stored in files based on HEALPixel group (`HPIXGROUP`) and HEALPixel number (`HEALPIX`):

- `spectro/redux/MOUNTAIN/healpix/SURVEY/PROGRAM/HPIXGROUP/HEALPIX/`

Healpix group is defined as `HPIXGROUP = floor(HEALPIX/100)`

### Glossary
- **Spectra:** Flux vs. wavelength, including an error model (inverse variance), mask bits (0=good, non-zero encodes what went wrong), and a Resolution Matrix modeling the effective instrument resolution per fiber per wavelength.
- **Survey:** DESI observations are split into multiple “survey” phases: Commissioning (`cmx`), Survey Validation (`sv1`, `sv2`, `sv3`), Main, and Special.
- **Program:** DESI Surveys are split into sub-programs (`dark`, `bright`, `backup`, `other`) depending upon the observing conditions.

### Read more:
- Data organization: https://data.desi.lbl.gov/doc/organization/
- Glossary: https://data.desi.lbl.gov/doc/glossary/
- Column descriptions: https://desidatamodel.readthedocs.io/en/latest/column_descriptions.html
- DR1 info: https://data.desi.lbl.gov/doc/releases/dr1/


## 01 - Locate the data

**Where is the data?** NERSC, at `/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/`
- **Core redshift catalog** is `zcatalog/v1/zall-pix-iron.fits`
- **Corresponding spectra** are in FITS files under `healpix/SURVEY/PROGRAM/GROUP/HPIX/coadd-SURVEY-PROGRAM-HPIX.fits`

In [3]:
! printf "MAIN DIRECTORY:\n"
! tree -L 1 $main_directory

! printf "\nCORE REDSHIFT CATALOG:\n"
! [ -f $core_redshift_catalog ] && echo "File exists: $core_redshift_catalog" || echo "File does not exist: $core_redshift_catalog"

! printf "\nCORRESPONDING SPECTRA DIRECTORY:\n"
! tree -L 2 $spectra_directory

MAIN DIRECTORY:
/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron
├── calibnight
├── exposures
├── exposures-iron.csv
├── exposures-iron.fits
├── exposure_tables
├── healpix
├── inventory-iron.txt
├── preproc
├── processing_tables
├── redux_iron.sha256sum
├── run
├── tiles
├── tiles-iron.csv
├── tiles-iron.fits
└── zcatalog

9 directories, 6 files

CORE REDSHIFT CATALOG:
File exists: /global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/zcatalog/v1/zall-pix-iron.fits

CORRESPONDING SPECTRA DIRECTORY:
/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/healpix
├── cmx
│   └── other
├── main
│   ├── backup
│   ├── bright
│   └── dark
├── README-tilepix
├── redux_iron_healpix.sha256sum
├── special
│   ├── backup
│   ├── bright
│   ├── dark
│   └── other
├── sv1
│   ├── backup
│   ├── bright
│   ├── dark
│   └── other
├── sv2
│   ├── backup
│   ├── bright
│   └── dark
├── sv3
│   ├── backup
│   ├── bright
│   └── dark
├── tilepix.fits
├── tilepix.json
├── tilepix-v0.fits
└── tilepix-v0.

**Important files:**
- `tiles-iron.fits` (or .csv) -- information about the observed tiles (telescope pointings with fibers assigned to specific targets)
- `exposures-iron.fits` (or .csv) -- information about individual exposures of tiles
- `zcatalog/` Directory -- redshift catalogs
- `tiles/` Directory -- spectra, coadds, and redshifts, grouped per-tile
- `healpix/` Directory -- spectra, coadds, and redshifts grouped by sky location (healpix)


## 02 - Figure out how the data is stored

Entirely fits, I guess...

From my notes:
> The corresponding spectra are in FITS files under `healpix/SURVEY/PROGRAM/GROUP/HPIX/coadd-SURVEY-PROGRAM-HPIX.fits`, where
>	- SURVEY, PROGRAM = columns in the redshift catalog for various subsets/epochs of the DESI survey
>	- HPIX = nside 64 nested healpix number (for DR1 this is constant, not dynamic like HATS)
>	- GROUP = int(HPIX/100) to avoid having thousands of healpix directories at the same level

### A naive look at the fits

In [4]:
target_fits = core_redshift_catalog

hdul = fits.open(target_fits)
hdul.info()

Filename: /global/cfs/cdirs/desi/public/dr1/spectro/redux/iron//zcatalog/v1/zall-pix-iron.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       8   ()      
  1  ZCATALOG      1 BinTableHDU    400   28425963R x 136C   [K, 7A, 6A, J, J, D, D, K, D, 10D, K, 6A, 20A, K, D, J, D, D, E, E, E, K, B, 3A, D, J, I, 8A, J, J, 4A, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, I, E, E, E, E, K, 2A, E, E, E, E, 1A, K, K, K, K, K, K, K, K, K, K, K, K, K, K, K, K, K, K, K, D, D, I, E, I, I, E, E, E, E, E, D, E, D, E, D, D, D, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, I, L, I, L, I, L, 22A]   


In [5]:
t = Table.read(core_redshift_catalog, hdu='ZCATALOG')
t

TARGETID,SURVEY,PROGRAM,HEALPIX,SPGRPVAL,Z,ZERR,ZWARN,CHI2,COEFF,NPIXELS,SPECTYPE,SUBTYPE,NCOEFF,DELTACHI2,COADD_FIBERSTATUS,TARGET_RA,TARGET_DEC,PMRA,PMDEC,REF_EPOCH,FA_TARGET,FA_TYPE,OBJTYPE,SUBPRIORITY,OBSCONDITIONS,RELEASE,BRICKNAME,BRICKID,BRICK_OBJID,MORPHTYPE,EBV,FLUX_G,FLUX_R,FLUX_Z,FLUX_W1,FLUX_W2,FLUX_IVAR_G,FLUX_IVAR_R,FLUX_IVAR_Z,FLUX_IVAR_W1,FLUX_IVAR_W2,FIBERFLUX_G,FIBERFLUX_R,FIBERFLUX_Z,FIBERTOTFLUX_G,FIBERTOTFLUX_R,FIBERTOTFLUX_Z,MASKBITS,SERSIC,SHAPE_R,SHAPE_E1,SHAPE_E2,REF_ID,REF_CAT,GAIA_PHOT_G_MEAN_MAG,GAIA_PHOT_BP_MEAN_MAG,GAIA_PHOT_RP_MEAN_MAG,PARALLAX,PHOTSYS,PRIORITY_INIT,NUMOBS_INIT,CMX_TARGET,DESI_TARGET,BGS_TARGET,MWS_TARGET,SCND_TARGET,SV1_DESI_TARGET,SV1_BGS_TARGET,SV1_MWS_TARGET,SV1_SCND_TARGET,SV2_DESI_TARGET,SV2_BGS_TARGET,SV2_MWS_TARGET,SV2_SCND_TARGET,SV3_DESI_TARGET,SV3_BGS_TARGET,SV3_MWS_TARGET,SV3_SCND_TARGET,PLATE_RA,PLATE_DEC,COADD_NUMEXP,COADD_EXPTIME,COADD_NUMNIGHT,COADD_NUMTILE,MEAN_DELTA_X,RMS_DELTA_X,MEAN_DELTA_Y,RMS_DELTA_Y,MEAN_PSF_TO_FIBER_SPECFLUX,MEAN_FIBER_RA,STD_FIBER_RA,MEAN_FIBER_DEC,STD_FIBER_DEC,MIN_MJD,MAX_MJD,MEAN_MJD,TSNR2_GPBDARK_B,TSNR2_ELG_B,TSNR2_GPBBRIGHT_B,TSNR2_LYA_B,TSNR2_BGS_B,TSNR2_GPBBACKUP_B,TSNR2_QSO_B,TSNR2_LRG_B,TSNR2_GPBDARK_R,TSNR2_ELG_R,TSNR2_GPBBRIGHT_R,TSNR2_LYA_R,TSNR2_BGS_R,TSNR2_GPBBACKUP_R,TSNR2_QSO_R,TSNR2_LRG_R,TSNR2_GPBDARK_Z,TSNR2_ELG_Z,TSNR2_GPBBRIGHT_Z,TSNR2_LYA_Z,TSNR2_BGS_Z,TSNR2_GPBBACKUP_Z,TSNR2_QSO_Z,TSNR2_LRG_Z,TSNR2_GPBDARK,TSNR2_ELG,TSNR2_GPBBRIGHT,TSNR2_LYA,TSNR2_BGS,TSNR2_GPBBACKUP,TSNR2_QSO,TSNR2_LRG,MAIN_NSPEC,MAIN_PRIMARY,SV_NSPEC,SV_PRIMARY,ZCAT_NSPEC,ZCAT_PRIMARY,DESINAME
,,,,,,,,,,,,,,,,deg,deg,mas / yr,mas / yr,yr,,,,,,,,,,,mag,nanomaggy,nanomaggy,nanomaggy,nanomaggy,nanomaggy,nanomaggy^-2,nanomaggy^-2,nanomaggy^-2,nanomaggy^-2,nanomaggy^-2,nanomaggy,nanomaggy,nanomaggy,nanomaggy,nanomaggy,nanomaggy,,,arcsec,,,,,mag,mag,mag,mas,,,,,,,,,,,,,,,,,,,,,deg,deg,,s,,,mm,mm,mm,mm,,deg,arcsec,deg,arcsec,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
int64,bytes7,bytes6,int32,int32,float64,float64,int64,float64,float64[10],int64,bytes6,bytes20,int64,float64,int32,float64,float64,float32,float32,float32,int64,uint8,bytes3,float64,int32,int16,bytes8,int32,int32,bytes4,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int16,float32,float32,float32,float32,int64,bytes2,float32,float32,float32,float32,bytes1,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,float64,float64,int16,float32,int16,int16,float32,float32,float32,float32,float32,float64,float32,float64,float32,float64,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int16,bool,int16,bool,int16,bool,bytes22
39628473198710603,cmx,other,2152,2152,0.8042057637744875,9.589004434794145e-06,0,10003.721982955933,117.8692939130018 .. 5.6065445634658255,7928,GALAXY,--,10,6319.3509159088135,0,23.764862479118218,29.832378962505196,0.0,0.0,2020.9597,3072,1,TGT,0.20970260241473437,3,9010,0237p297,494512,3915,EXP,0.05433502,1.3540096,2.9762945,10.3957405,27.52511,48.7791,531.4606,123.64128,19.293917,2.6645064,0.6410277,0.4344953,0.95507884,3.3359442,0.43980038,0.9776775,3.4928923,0,1.0,1.1636901,0.18920432,-0.24198028,0,--,0.0,0.0,0.0,0.0,S,3200,1,3072,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,23.764862479118218,29.832378962505196,4,3600.0,1,1,-0.00675,0.0091515025,-0.0055,0.014124447,0.71950334,23.76489350625343,0.08943564,29.832358617984326,0.1715876,59200.06640136,59200.12381137,59200.095110124996,388.2487,0.28193328,75.31641,283.94504,1451.2411,530.06067,7.442073,2.1379867,34921.81,78.42465,6280.0347,0.1178768,7149.4326,36527.004,22.981281,113.36229,5.09169e-05,260.3999,9.6547765e-06,0.0,11533.531,6.420885e-05,52.333626,120.35188,35310.06

### Using the DESI Getting Started Tutorials

> Since this is a rather large file (136 columns by 28 million rows), we will use fitsio to read just a subset of the columns.  
> After reading, we'll wrap this into a Table object for nicer printing.  
> Even with a subset of columns, this can take 30-60 seconds to read.

In [6]:
if 'DESI_ROOT' not in os.environ:
    print("ERROR: Ideally $DESI_ROOT should be set to data location before starting Jupyter")
    if 'NERSC_HOST' in os.environ:
        print('Setting $DESI_ROOT to NERSC DR1 location')
        os.environ['DESI_ROOT'] = '/global/cfs/cdirs/desi/public/dr1'
    else:
        print("ERROR: update this cell to set $DESI_ROOT to your local location of DESI data")
        # os.environ['DESI_ROOT'] = '/path/to/local/desi/tiny_dr1'

ERROR: Ideally $DESI_ROOT should be set to data location before starting Jupyter
Setting $DESI_ROOT to NERSC DR1 location


> Since this is a rather large file (136 columns by 28 million rows), we will use fitsio to read just a subset of the columns.  
> After reading, we'll wrap this into a Table object for nicer printing. Even with a subset of columns, this can take 30-60 seconds to read.

In [ ]:
# Note - this cell can take ~10 minutes.

# Release directory path
specprod = 'iron'    # Primary spectroscopic production in DR1
desi_root = os.environ['DESI_ROOT']
specprod_dir = f'{desi_root}/spectro/redux/{specprod}'
print(specprod_dir)

# Read fits
columns = ['TARGETID', 'SURVEY', 'PROGRAM', 'DESI_TARGET', 'Z', 'SPECTYPE', 'ZWARN', 'DELTACHI2', 'FLUX_G', 'FLUX_R', 'FLUX_Z', 'ZCAT_PRIMARY', 'HEALPIX']
zcat = Table(fitsio.read(f'{specprod_dir}/zcatalog/v1/zall-pix-{specprod}.fits', "ZCATALOG", columns=columns))

/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron


> Summarizing the columns that we read:
- `TARGETID`: a unique integer for each DESI target  
- `SURVEY`: DESI operations are split into different survey phases (e.g. main, sv1, sv3)  
- `PROGRAM`: a subdivision of SURVEY, split by observing conditions (e.g. dark, bright)  
- `DESI_TARGET`: a bit mask recording why each target was selected for observation  
- `Z`: the measured redshift  
- `SPECTYPE`: the measured spectral type (e.g. GALAXY, QSO, STAR)  
- `ZWARN`: a bit mask flag of redshift/spectype fitting problems; 0 is good  
- `DELTACHI2`: the difference in the chi2 of the best fit to the next best fit; bigger is better meaning that the best fit is more confidently the single correct solution  
- `FLUX_G/R/Z`: object fluxes from imaging (magnitude = 22.5 - 2.5*log10(flux))  
- `ZCAT_PRIMARY`: DESI targets can be observed more than once on different tiles / surveys / programs, and thus appear multiple times in a catalog. This flag indicates which entry is likely the best measurement.  
- `HEALPIX`: which nside=64 nested healpixel this target is on (used for finding the actual spectrum on disk)

In [8]:
print(f"Raw number of zcat entries: {len(zcat):,}")
for spectype in ['GALAXY', 'QSO', 'STAR']:
    print(f"Raw number of zcat entries for spectype '{spectype}': {np.sum(zcat['SPECTYPE']==spectype):,}")

Raw number of zcat entries: 28,425,963
Raw number of zcat entries for spectype 'GALAXY': 21,696,490


Raw number of zcat entries for spectype 'QSO': 1,862,583
Raw number of zcat entries for spectype 'STAR': 4,866,890


In [9]:
# Surveys and programs

# print header of possible programs
print('        ', end='')
programs = np.unique(zcat['PROGRAM'])
for program in np.unique(zcat['PROGRAM']):
    print(f'{program:>12s}', end='')
print()

# print number of targets per survey/program
for survey in np.unique(zcat['SURVEY']):
    this_survey = (zcat['SURVEY'] == survey)
    print(f'{survey:8s}', end='')
    for program in programs:
        this_program = (zcat['PROGRAM'] == program)
        n = np.sum(this_survey & this_program)
        print(f'{n:12,}', end='')
    print('')

              backup      bright        dark       other
cmx                0           0           0       5,000
main       1,632,500  11,020,470  12,778,525           0
special       44,905      74,412      19,500      64,428
sv1          110,599     239,057     371,000     143,679
sv2            4,985      82,288      85,411           0
sv3          156,359     729,898     862,947           0


## 03 - Get the data
- Best case scenario: take everything
- However, need to know how much storage it would take
- And we were given the 50 TB honor system quota
- So, might want to take a chunk of only 10% first

### General exploration

In [4]:
# Check size of redshift catalog fits file (core_redshift_catalog):
file_size_bytes = os.path.getsize(core_redshift_catalog)
file_size_gb = file_size_bytes / (1024 ** 3)
print(f"Size of {core_redshift_catalog}: {file_size_gb:.2f} GB")

Size of /global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/zcatalog/v1/zall-pix-iron.fits: 20.83 GB


But it's the added spectra files that are way bigger. Looking into them for a while, they're on the order of at least a few TB.

**Let's look into the "tiny DR1" dataset they reference in the DR1 docs.**

You'd get it via:
` curl https://raw.githubusercontent.com/desihub/desida/refs/tags/1.0.0/bin/desi_get_dr_subset > desi_get_dr_subset
python desi_get_dr_subset`

> NOTE: even a "tiny" subset of DR1 is ~40 GB and can take an hour or more to download 
> even with a fast internet connection. 

Though, I can't find much more about it--just references towards using it for the tutorials.

Maybe if we just look at survey=main, program=dark?

In [ ]:
# Check the size of the spectra directory:
if False:
    total_size_bytes = 0
    target_directory = spectra_directory + "main/dark"
    with tqdm(desc="Scanning files", unit=" files") as pbar:
        for dirpath, dirnames, filenames in os.walk(target_directory):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                total_size_bytes += os.path.getsize(fp)
                pbar.update(1)
                pbar.set_postfix(size_tb=f"{total_size_bytes / (1024 ** 4):.2f}")

    total_size_gb = total_size_bytes / (1024 ** 3)
    print(f"Total size of target directory {target_directory}: {total_size_gb:.2f} GB")

# Running this for half an hour before interrupting:
# Scanning files: 185625 files [30:52, 100.20 files/s, size_tb=12.64]

### Focus on a subset

Ok, this is still really big. Let's just focus in on a very specific subset. The directory tree looks like this:

```
CORRESPONDING SPECTRA DIRECTORY:
/global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/healpix/
├── cmx
│   └── other
├── main
│   ├── backup
│   ├── bright
│   └── dark
│       └── 0
│           └── 0
|               ├── coadd-main-dark-0.fits
|               ├── emline-main-dark-0.fits
|               ├── hpixexp-main-dark-0.csv
|               ├── logs
|               │   ├── emline-main-dark-0.log
|               │   ├── qso_mgii-main-dark-0.log
|               │   ├── qso_qn-main-dark-0.log
|               │   ├── redrock-main-dark-0.log
|               │   ├── redux_iron_healpix_main_dark_0_0_logs.sha256sum
|               │   └── spectra-main-dark-0.log
|               ├── qso_mgii-main-dark-0.fits
|               ├── qso_qn-main-dark-0.fits
|               ├── redrock-main-dark-0.fits
|               ├── redux_iron_healpix_main_dark_0_0.sha256sum
|               ├── rrdetails-main-dark-0.h5
|               └── spectra-main-dark-0.fits.gz
│           └── ...
│       └── ...
├── ...
```

Which, when we ask claude about, gives us:
| File | Description |
|------|-------------|
| `spectra-main-dark-0.fits.gz` | Per-exposure spectra for each target: flux, wavelength, inverse variance, and mask |
| `coadd-main-dark-0.fits` | Coadded spectra (multiple exposures combined per target); more convenient than `spectra` for most analyses |
| `redrock-main-dark-0.fits` | Redshift catalog from Redrock: `Z`, `ZWARN`, `DELTACHI2`, `SPECTYPE`, etc. |
| `rrdetails-main-dark-0.h5` | Detailed Redrock fitting outputs (full chi2 surfaces); only needed if digging into redshift fitting |
| `emline-main-dark-0.fits` | Emission line fits; useful if you need line fluxes or equivalent widths |
| `qso_mgii-main-dark-0.fits` | MgII QSO classifier outputs; relevant only for quasar work |
| `qso_qn-main-dark-0.fits` | QuasarNet QSO classifier outputs; relevant only for quasar work |
| `hpixexp-main-dark-0.csv` | Per-healpixel exposure information |
| `redux_iron_healpix_main_dark_0_0.sha256sum` | Checksums for verifying file integrity |

Where, according to the [DESI Glossary](https://data.desi.lbl.gov/doc/glossary/), 
- **Redrock:** The spectroscopic classification and redshift fitting pipeline for DESI spectra.

Looks like we'll want to focus on:
- **spectra-main-dark**
- **redrock-main-dark**
- maybe **hpix-exp-main-dark**, if it has things like the seeing, and if those are columns we'd like to add

#### spectra-main-dark

In [18]:
target_fits = spectra_directory / "main/dark/0/0/spectra-main-dark-0.fits.gz"

with fits.open(target_fits) as hdul:
    hdul.info()

Filename: /global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/healpix/main/dark/0/0/spectra-main-dark-0.fits.gz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     156   ()      
  1  FIBERMAP      1 BinTableHDU    209   2356R x 75C   [K, I, J, K, J, J, D, D, E, E, E, E, K, B, 3A, E, E, J, D, J, I, 8A, J, J, 4A, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, I, E, E, E, E, K, 2A, E, E, E, E, 1A, K, K, K, K, K, K, D, D, K, D, D, D, D, D, D, D, D, J, J, D, J]   
  2  B_WAVELENGTH    1 ImageHDU        10   (2751,)   float64   
  3  B_FLUX        1 ImageHDU        11   (2751, 2356)   float32   
  4  B_IVAR        1 ImageHDU        11   (2751, 2356)   float32   
  5  B_MASK        1 ImageHDU        10   (2751, 2356)   int32   
  6  B_RESOLUTION    1 ImageHDU        11   (2751, 11, 2356)   float32   
  7  R_WAVELENGTH    1 ImageHDU        10   (2326,)   float64   
  8  R_FLUX        1 ImageHDU        11   (2326, 2356)   float32   
  9  R_IVA

Our agentic friend tells us the structure is:

- **`FIBERMAP`**: the target catalog table — 2356 targets × 75 columns. This is where `TARGETID`, `RA`, `DEC`, and other target metadata live.
- **`B/R/Z_WAVELENGTH`**: 1D wavelength arrays for each camera (B, R, Z) — shared across all targets.
- **`B/R/Z_FLUX`**: 2D flux arrays, shape `(n_wavelengths, n_targets)` — so each row is a wavelength bin, each column is a target.
- **`B/R/Z_IVAR`**: inverse variance (1/noise²) — same shape as flux.
- **`B/R/Z_MASK`**: bitmask flagging bad pixels — same shape as flux.
- **`B/R/Z_RESOLUTION`**: resolution matrix, shape `(n_wavelengths, 11, n_targets)` — needed for precise line fitting, probably not critical for the import.
- **`SCORES`**: per-spectrum quality scores, 2356 rows × 61 columns.

In [ ]:
# Let's look at the spectra-main FIBERMAP columns:

with fits.open(target_fits) as hdul:
    fibermap = hdul['FIBERMAP']
    for i, line in enumerate(fibermap.columns):
        if i < 5:
            print(line)
    print("... ")
    print(f"Total number of columns in FIBERMAP: {len(fibermap.columns)}")

name = 'TARGETID'; format = 'K'
name = 'PETAL_LOC'; format = 'I'
name = 'DEVICE_LOC'; format = 'J'
name = 'LOCATION'; format = 'K'
name = 'FIBER'; format = 'J'
... 
Total number of columns in FIBERMAP: 75


| Column(s) | Source | Notes |
|-----------|--------|-------|
| `TARGETID` | FIBERMAP | Unique target identifier; join key between files |
| `TARGET_RA`, `TARGET_DEC` | FIBERMAP | Sky coordinates for the HATS spatial index |
| `DESI_TARGET`, `BGS_TARGET`, `MWS_TARGET` | FIBERMAP | Target selection bitmasks |
| `FLUX_G/R/Z/W1/W2`, `FLUX_IVAR_G/R/Z/W1/W2` | FIBERMAP | Imaging photometry |
| `EBV` | FIBERMAP | Dust extinction |
| `MORPHTYPE` | FIBERMAP | Galaxy morphology |
| `PHOTSYS` | FIBERMAP | Photometric system (N/S) |
| `MJD`, `EXPTIME`, `NIGHT` | FIBERMAP | Observation metadata |
| `Z`, `ZWARN`, `DELTACHI2`, `SPECTYPE` | redrock | Redshift results; not in FIBERMAP |
| `HEALPIX` | redrock | Can be inferred from filename but cleaner to get explicitly |

#### coadd-main-dark

In [5]:
target_fits = spectra_directory / "main/dark/0/0/coadd-main-dark-0.fits"

with fits.open(target_fits) as hdul:
    hdul.info()

Filename: /global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/healpix/main/dark/0/0/coadd-main-dark-0.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     157   ()      
  1  FIBERMAP      1 BinTableHDU    193   1704R x 67C   [K, J, D, D, E, E, E, K, B, 3A, D, J, I, 8A, J, J, 4A, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, I, E, E, E, E, K, 2A, E, E, E, E, 1A, K, K, K, K, K, K, D, D, I, E, I, I, E, E, E, E, D, E, D, E, E]   
  2  EXP_FIBERMAP    1 BinTableHDU     64   2356R x 26C   [K, J, D, J, J, D, J, D, I, J, K, J, J, E, E, E, D, D, K, D, D, D, D, D, D, D]   
  3  B_WAVELENGTH    1 ImageHDU        10   (2751,)   float64   
  4  B_FLUX        1 ImageHDU        11   (2751, 1704)   float32   
  5  B_IVAR        1 ImageHDU        11   (2751, 1704)   float32   
  6  B_MASK        1 ImageHDU        10   (2751, 1704)   int32   
  7  B_RESOLUTION    1 ImageHDU        11   (2751, 11, 1704)   float32   
  8  R_WAVELENGTH    1 ImageHDU 

In [6]:
with fits.open(target_fits, memmap=True) as hdul:
    print(hdul['FIBERMAP'].columns)

ColDefs(
    name = 'TARGETID'; format = 'K'
    name = 'COADD_FIBERSTATUS'; format = 'J'
    name = 'TARGET_RA'; format = 'D'
    name = 'TARGET_DEC'; format = 'D'
    name = 'PMRA'; format = 'E'
    name = 'PMDEC'; format = 'E'
    name = 'REF_EPOCH'; format = 'E'
    name = 'FA_TARGET'; format = 'K'
    name = 'FA_TYPE'; format = 'B'
    name = 'OBJTYPE'; format = '3A'
    name = 'SUBPRIORITY'; format = 'D'
    name = 'OBSCONDITIONS'; format = 'J'
    name = 'RELEASE'; format = 'I'
    name = 'BRICKNAME'; format = '8A'
    name = 'BRICKID'; format = 'J'
    name = 'BRICK_OBJID'; format = 'J'
    name = 'MORPHTYPE'; format = '4A'
    name = 'EBV'; format = 'E'
    name = 'FLUX_G'; format = 'E'
    name = 'FLUX_R'; format = 'E'
    name = 'FLUX_Z'; format = 'E'
    name = 'FLUX_W1'; format = 'E'
    name = 'FLUX_W2'; format = 'E'
    name = 'FLUX_IVAR_G'; format = 'E'
    name = 'FLUX_IVAR_R'; format = 'E'
    name = 'FLUX_IVAR_Z'; format = 'E'
    name = 'FLUX_IVAR_W1'; format = 'E'


#### redrock-main-dark

In [23]:
target_fits = spectra_directory / "main/dark/0/0/redrock-main-dark-0.fits"

with fits.open(target_fits) as hdul:
    hdul.info()

Filename: /global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/healpix/main/dark/0/0/redrock-main-dark-0.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      85   ()      
  1  REDSHIFTS     1 BinTableHDU     32   1704R x 11C   [K, D, D, K, D, 10D, K, 6A, 20A, K, D]   
  2  FIBERMAP      1 BinTableHDU    143   1704R x 67C   [K, J, D, D, E, E, E, K, B, 3A, D, J, I, 8A, J, J, 4A, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, I, E, E, E, E, K, 2A, E, E, E, E, 1A, K, K, K, K, K, K, D, D, I, E, I, I, E, E, E, E, D, E, D, E, E]   
  3  EXP_FIBERMAP    1 BinTableHDU     61   2356R x 26C   [K, J, D, J, J, D, J, D, I, J, K, J, J, E, E, E, D, D, K, D, D, D, D, D, D, D]   
  4  TSNR2         1 BinTableHDU     75   1704R x 33C   [K, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E]   


| HDU | Rows | Columns | Notes |
|-----|------|---------|-------|
| `REDSHIFTS` | 1704 | 11 | Core redshift results: `Z`, `ZWARN`, `DELTACHI2`, `SPECTYPE` etc. Fewer rows than spectra (1704 vs 2356) — redrock fits coadded/primary targets only |
| `FIBERMAP` | 1704 | 67 | Per-target metadata, similar to spectra FIBERMAP but slightly different column set |
| `EXP_FIBERMAP` | 2356 | 26 | Per-exposure fibermap — matches the spectra file row count |
| `TSNR2` | 1704 | 33 | Template S/N² per target; useful for quality cuts |

In [27]:
# Let's look at the redrock-main REDSHIFTS columns:

with fits.open(target_fits) as hdul:
    redshifts = hdul['REDSHIFTS']
    for _, line in enumerate(redshifts.columns):
        print(line)
    print(f"\nTotal number of columns in REDSHIFTS: {len(redshifts.columns)}")

name = 'TARGETID'; format = 'K'
name = 'Z'; format = 'D'
name = 'ZERR'; format = 'D'
name = 'ZWARN'; format = 'K'
name = 'CHI2'; format = 'D'
name = 'COEFF'; format = '10D'; dim = '(10)'
name = 'NPIXELS'; format = 'K'
name = 'SPECTYPE'; format = '6A'
name = 'SUBTYPE'; format = '20A'
name = 'NCOEFF'; format = 'K'
name = 'DELTACHI2'; format = 'D'

Total number of columns in REDSHIFTS: 11


| Column | Format | Notes |
|--------|--------|-------|
| `TARGETID` | int64 | Join key to FIBERMAP in spectra file |
| `Z` | float64 | Best-fit redshift |
| `ZERR` | float64 | Redshift uncertainty |
| `ZWARN` | int64 | Warning bitmask; 0 = good fit |
| `CHI2` | float64 | Chi² of best fit |
| `DELTACHI2` | float64 | Chi² difference between best and second-best fit; larger = more confident |
| `NPIXELS` | int64 | Number of pixels used in the fit |
| `SPECTYPE` | string | Spectral type: `GALAXY`, `QSO`, or `STAR` |
| `SUBTYPE` | string | More specific subtype (e.g. `CV` for cataclysmic variables) |
| `NCOEFF` | int64 | Number of template coefficients used |
| `COEFF` | float64 × 10 | Template coefficients — probably skip for HATS import |

#### zall-pix-iron, the core redshift catalog
Note that this one is for everything, and not stored in the /group/healpix dirs

In [18]:
with fits.open(core_redshift_catalog) as hdul:
    hdul.info()

Filename: /global/cfs/cdirs/desi/public/dr1/spectro/redux/iron/zcatalog/v1/zall-pix-iron.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       8   ()      
  1  ZCATALOG      1 BinTableHDU    400   28425963R x 136C   [K, 7A, 6A, J, J, D, D, K, D, 10D, K, 6A, 20A, K, D, J, D, D, E, E, E, K, B, 3A, D, J, I, 8A, J, J, 4A, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, I, E, E, E, E, K, 2A, E, E, E, E, 1A, K, K, K, K, K, K, K, K, K, K, K, K, K, K, K, K, K, K, K, D, D, I, E, I, I, E, E, E, E, E, D, E, D, E, D, D, D, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, I, L, I, L, I, L, 22A]   


In [21]:
# Let's look at the ZCATALOG columns:

with fits.open(core_redshift_catalog) as hdul:
    zcatalog = hdul['ZCATALOG']
    for i, line in enumerate(zcatalog.columns):
        if i < 5:
            print(line)
    print("... ")
    print(f"\nTotal number of columns in ZCATALOG: {len(zcatalog.columns)}")

name = 'TARGETID'; format = 'K'
name = 'SURVEY'; format = '7A'
name = 'PROGRAM'; format = '6A'
name = 'HEALPIX'; format = 'J'
name = 'SPGRPVAL'; format = 'J'
... 

Total number of columns in ZCATALOG: 136
